# From a Low-Pass Filter to a Kalman Filter

This notebook builds the idea in stages:

1. Create a known signal and noisy measurements.
2. Smooth the measurements with a first-order low-pass filter.
3. Introduce prediction uncertainty and measurement uncertainty.
4. Implement a numerically robust linear Kalman filter.
5. Scale from a one-state model to a two-state position/velocity model.

The code uses column vectors and matrices consistently. A one-dimensional
filter is therefore just the smallest case of the same implementation used
for larger systems.

**Dependencies:** NumPy and Matplotlib.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


## 1. Create a reproducible measurement problem

A useful test separates the unknown true state from its measurement:

$$z_k = x_k + v_k, \qquad v_k \sim \mathcal{N}(0, R).$$

The noise is additive and zero-mean. Multiplying the true signal by a random
number would instead create biased, signal-dependent noise and would make the
filter comparison misleading.


In [ ]:
rng = np.random.default_rng(7)

dt = 0.1                         # seconds between samples
sample_count = 500
time = np.arange(sample_count) * dt

truth = np.cos(0.55 * time) + 0.15 * np.sin(0.12 * time)
measurement_std = 0.28
measurements = truth + rng.normal(0.0, measurement_std, sample_count)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(time, truth, label="true state", linewidth=2)
ax.scatter(time, measurements, label="sensor measurement", s=9, alpha=0.35)
ax.set(xlabel="time (s)", ylabel="value", title="Truth and noisy measurements")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


## 2. First-order low-pass filter

For sampling interval $\Delta t$ and time constant $\tau$,

$$\alpha = 1-e^{-\Delta t/\tau}$$

and the update is

$$\hat{x}_k = \hat{x}_{k-1}+\alpha(z_k-\hat{x}_{k-1}).$$

A larger $\tau$ gives more smoothing and more lag. Notice that $\Delta t$ is
the **sampling interval**; the sample rate is $f_s=1/\Delta t$.


In [ ]:
def low_pass_filter(values, dt, tau, initial=None):
    if dt <= 0 or tau <= 0:
        raise ValueError("dt and tau must be positive")

    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or values.size == 0:
        raise ValueError("values must be a non-empty one-dimensional array")

    alpha = 1.0 - np.exp(-dt / tau)
    estimate = values[0] if initial is None else float(initial)
    filtered = np.empty_like(values)

    for index, measurement in enumerate(values):
        estimate += alpha * (measurement - estimate)
        filtered[index] = estimate

    return filtered


low_pass = low_pass_filter(measurements, dt=dt, tau=0.6)


## 3. What the Kalman filter adds

A low-pass filter uses a fixed weight. A Kalman filter computes its weight
from two uncertainty models:

- $Q$: process covariance—how much the real state can depart from the model.
- $R$: measurement covariance—how noisy the sensor is after bias calibration.
- $P$: state-estimation error covariance—the filter's current uncertainty.

For a linear system,

$$x_k = F x_{k-1} + B u_k + w_k, \qquad w_k\sim\mathcal{N}(0,Q)$$
$$z_k = H x_k + v_k, \qquad v_k\sim\mathcal{N}(0,R).$$

The two stages are:

**Predict**

$$\hat{x}_k^- = F\hat{x}_{k-1}+Bu_k$$
$$P_k^- = FP_{k-1}F^T+Q$$

**Update**

$$\tilde{y}_k=z_k-H\hat{x}_k^-$$
$$S_k=HP_k^-H^T+R$$
$$K_k=P_k^-H^TS_k^{-1}$$
$$\hat{x}_k=\hat{x}_k^-+K_k\tilde{y}_k.$$

The implementation below solves the gain equation without explicitly forming
an inverse and uses the Joseph covariance update for numerical stability.


In [ ]:
def _matrix(value, shape, name):
    array = np.asarray(value, dtype=float)
    if array.shape != shape:
        raise ValueError(f"{name} must have shape {shape}; received {array.shape}")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{name} contains a non-finite value")
    return array.copy()


def _column(value, rows, name):
    array = np.asarray(value, dtype=float)
    if array.ndim == 0:
        array = array.reshape(1, 1)
    elif array.ndim == 1:
        array = array.reshape(-1, 1)
    if array.shape != (rows, 1):
        raise ValueError(f"{name} must have shape {(rows, 1)}; received {array.shape}")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{name} contains a non-finite value")
    return array


class LinearKalmanFilter:
    '''Discrete linear Kalman filter with explicit matrix dimensions.'''

    def __init__(self, F, H, Q, R, x0, P0, B=None):
        F = np.asarray(F, dtype=float)
        H = np.asarray(H, dtype=float)
        if F.ndim != 2 or F.shape[0] != F.shape[1]:
            raise ValueError("F must be a square matrix")
        if H.ndim != 2 or H.shape[1] != F.shape[0]:
            raise ValueError("H must have one column per state")

        self.n_x = F.shape[0]
        self.n_z = H.shape[0]
        self.F = _matrix(F, (self.n_x, self.n_x), "F")
        self.H = _matrix(H, (self.n_z, self.n_x), "H")
        self.Q = _matrix(Q, (self.n_x, self.n_x), "Q")
        self.R = _matrix(R, (self.n_z, self.n_z), "R")
        self.x = _column(x0, self.n_x, "x0")
        self.P = _matrix(P0, (self.n_x, self.n_x), "P0")
        self.B = None if B is None else np.asarray(B, dtype=float)

        if self.B is not None and (self.B.ndim != 2 or self.B.shape[0] != self.n_x):
            raise ValueError("B must have one row per state")

    def predict(self, u=None):
        self.x = self.F @ self.x
        if self.B is not None:
            if u is None:
                raise ValueError("u is required because B was supplied")
            self.x += self.B @ _column(u, self.B.shape[1], "u")
        elif u is not None:
            raise ValueError("u was supplied but B is not configured")

        self.P = self.F @ self.P @ self.F.T + self.Q
        self.P = 0.5 * (self.P + self.P.T)
        return self.x.copy(), self.P.copy()

    def update(self, z):
        z = _column(z, self.n_z, "z")
        innovation = z - self.H @ self.x
        innovation_covariance = self.H @ self.P @ self.H.T + self.R

        # Solve S @ K.T = (P @ H.T).T instead of computing inv(S).
        state_measurement_covariance = self.P @ self.H.T
        gain = np.linalg.solve(
            innovation_covariance,
            state_measurement_covariance.T,
        ).T

        self.x = self.x + gain @ innovation

        identity = np.eye(self.n_x)
        residual_map = identity - gain @ self.H
        self.P = (
            residual_map @ self.P @ residual_map.T
            + gain @ self.R @ gain.T
        )
        self.P = 0.5 * (self.P + self.P.T)

        return self.x.copy(), self.P.copy(), innovation.copy(), gain.copy()


## 4. One-state Kalman filter

The smallest useful model says the next value will be approximately the
current value:

$$x_k=x_{k-1}+w_k, \qquad z_k=x_k+v_k.$$

Thus $F=H=[1]$. This is a **random-walk model**, not a physical model of the
cosine signal. It behaves like an adaptive low-pass filter and is a good
place to learn the mechanics.


In [ ]:
random_walk_filter = LinearKalmanFilter(
    F=np.array([[1.0]]),
    H=np.array([[1.0]]),
    Q=np.array([[0.01]]),
    R=np.array([[measurement_std**2]]),
    x0=np.array([[measurements[0]]]),
    P0=np.array([[1.0]]),
)

kalman_1d = []
gains_1d = []
variances_1d = []

for measurement in measurements:
    random_walk_filter.predict()
    state, covariance, innovation, gain = random_walk_filter.update(measurement)
    kalman_1d.append(state.item())
    gains_1d.append(gain.item())
    variances_1d.append(covariance.item())

kalman_1d = np.asarray(kalman_1d)
gains_1d = np.asarray(gains_1d)
variances_1d = np.asarray(variances_1d)


## 5. Scale the model: position and velocity

A better model stores two states:

$$x = \begin{bmatrix}\text{position}\\\text{velocity}\end{bmatrix},\qquad
  F = \begin{bmatrix}1&\Delta t\\0&1\end{bmatrix},\qquad
  H = \begin{bmatrix}1&0\end{bmatrix}.$$

The sensor measures only position, but the filter estimates both position
and velocity. The chosen $Q$ models unknown acceleration. This is the key
scalability benefit of the matrix formulation.


In [ ]:
acceleration_variance = 0.6
F_cv = np.array([[1.0, dt], [0.0, 1.0]])
H_cv = np.array([[1.0, 0.0]])
Q_cv = acceleration_variance * np.array([
    [dt**4 / 4.0, dt**3 / 2.0],
    [dt**3 / 2.0, dt**2],
])

constant_velocity_filter = LinearKalmanFilter(
    F=F_cv,
    H=H_cv,
    Q=Q_cv,
    R=np.array([[measurement_std**2]]),
    x0=np.array([[measurements[0]], [0.0]]),
    P0=np.diag([1.0, 1.0]),
)

kalman_2d = []
velocity_2d = []
for measurement in measurements:
    constant_velocity_filter.predict()
    state, covariance, innovation, gain = constant_velocity_filter.update(measurement)
    kalman_2d.append(state[0, 0])
    velocity_2d.append(state[1, 0])

kalman_2d = np.asarray(kalman_2d)
velocity_2d = np.asarray(velocity_2d)


## 6. Compare the filters

Root-mean-square error is useful here because the simulated truth is known.
In real flight data, inspect innovations, compare against independent truth
data when possible, and validate across the full operating envelope.


In [ ]:
def rmse(estimate, reference):
    estimate = np.asarray(estimate, dtype=float)
    reference = np.asarray(reference, dtype=float)
    return np.sqrt(np.mean((estimate - reference) ** 2))


scores = {
    "raw measurement": rmse(measurements, truth),
    "low-pass": rmse(low_pass, truth),
    "1-state Kalman": rmse(kalman_1d, truth),
    "2-state Kalman": rmse(kalman_2d, truth),
}

for name, score in scores.items():
    print(f"{name:>17}: RMSE = {score:.4f}")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time, truth, label="truth", linewidth=2.5, color="black")
ax.scatter(time, measurements, label="measurements", s=8, alpha=0.2)
ax.plot(time, low_pass, label="low-pass", linewidth=1.6)
ax.plot(time, kalman_1d, label="1-state Kalman", linewidth=1.6)
ax.plot(time, kalman_2d, label="2-state Kalman", linewidth=1.8)
ax.set(xlabel="time (s)", ylabel="value", title="Filter comparison")
ax.grid(alpha=0.25)
ax.legend(ncol=3)
plt.show()


## 7. Inspect filter confidence and gain

For the one-state filter, $P$ and $K$ start from their initial conditions and
converge toward steady values. A large gain moves strongly toward the new
measurement; a small gain trusts the prediction more.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(time, gains_1d)
axes[0].set(ylabel="Kalman gain", title="One-state filter diagnostics")
axes[1].plot(time, variances_1d)
axes[1].set(xlabel="time (s)", ylabel="posterior variance")
for axis in axes:
    axis.grid(alpha=0.25)
plt.show()


## 8. Estimating sensor covariance correctly

Stationary samples provide two different quantities:

- Their mean estimates **bias**. Remove it or include it in the state.
- Their variance around that mean estimates measurement noise $R$.

For a finite calibration sample, `ddof=1` gives the sample variance. Real
sensors should be characterized over temperature, voltage, time, and relevant
dynamic conditions. For multiple axes, $R$ may include cross-axis covariance.


In [ ]:
def estimate_scalar_sensor_noise(stationary_samples, known_stationary_value=0.0):
    samples = np.asarray(stationary_samples, dtype=float)
    if samples.ndim != 1 or samples.size < 2:
        raise ValueError("at least two scalar samples are required")

    bias = samples.mean() - known_stationary_value
    variance = samples.var(ddof=1)
    return bias, np.array([[variance]])


stationary_samples = np.array([0.00, -0.20, 0.30, 0.15, 0.16, -0.07, 0.12, -0.20])
estimated_bias, estimated_R = estimate_scalar_sensor_noise(stationary_samples)
print(f"estimated bias: {estimated_bias:.4f}")
print(f"estimated R: {estimated_R.item():.6f}")


## 9. Basic correctness checks

These checks do not prove that $F$, $Q$, or $R$ represent the real system,
but they catch common implementation failures.


In [ ]:
assert np.all(np.isfinite(kalman_2d))
assert np.allclose(constant_velocity_filter.P, constant_velocity_filter.P.T, atol=1e-12)
assert np.linalg.eigvalsh(constant_velocity_filter.P).min() >= -1e-12
assert np.all((gains_1d >= 0.0) & (gains_1d <= 1.0))
print("All numerical sanity checks passed.")


## 10. Tuning and ADCS guidance

- Increase $Q$ when the model misses real motion; the filter reacts faster to
  measurements but passes more noise.
- Increase $R$ when measurements are less reliable; the filter trusts the
  model more.
- Choose $Q$ from physical disturbance models or system identification, not
  only by visual smoothness.
- Validate innovations: they should be approximately zero-mean and consistent
  with the predicted innovation covariance $S$.
- Keep units explicit. Every entry of $P$, $Q$, and $R$ carries squared or
  cross-product units corresponding to its states and measurements.

### CubeSat attitude determination

A linear scalar filter should not be applied directly to quaternion
components. A common flight architecture is a multiplicative/error-state EKF:

- propagate a normalized nominal quaternion with gyroscope measurements;
- estimate a small attitude error and gyroscope bias, often
  $\delta x=[\delta\theta_x,\delta\theta_y,\delta\theta_z,b_x,b_y,b_z]^T$;
- maintain a $6\times6$ covariance;
- update with magnetometer, sun-sensor, or star-tracker vector observations;
- inject the small-angle correction into the quaternion, normalize it, and
  reset the attitude-error state.

The `LinearKalmanFilter` here is a sound learning base for matrix dimensions,
covariance propagation, and sensor fusion, but it is not itself a complete
flight ADCS estimator.
